In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [3]:
# device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [12]:
# Hyper-Parameters
num_epochs = 10
batch_size = 4
learning_rate = 0.001

In [13]:
# dataset has PILImage images of range [0, 1]
# We transform them to Tensor of normalized range [-1, 1]
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

In [14]:
train_dataset = torchvision.datasets.CIFAR10(root='data', train=True,
                                             download=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(root='data', train=False,
                                             download=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size,
                                           shuffle=False)

In [15]:
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

In [16]:
# implement conv net
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16*5*5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [17]:
model = ConvNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

n_total_steps = len(train_loader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # origin shape: [4, 3, 32, 32] = 4, 3, 1024
        # input_layer: 3 input channels, 6 output channels, 5 kernel size
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 2000 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{n_total_steps}], Loss: {loss.item():.4f}")
print("Finished training")

Epoch [1/10], Step [2000/12500], Loss: 2.2781
Epoch [1/10], Step [4000/12500], Loss: 2.2901
Epoch [1/10], Step [6000/12500], Loss: 2.3040
Epoch [1/10], Step [8000/12500], Loss: 2.3115
Epoch [1/10], Step [10000/12500], Loss: 2.2963
Epoch [1/10], Step [12000/12500], Loss: 2.2779
Epoch [2/10], Step [2000/12500], Loss: 1.9531
Epoch [2/10], Step [4000/12500], Loss: 1.9711
Epoch [2/10], Step [6000/12500], Loss: 1.3065
Epoch [2/10], Step [8000/12500], Loss: 1.4354
Epoch [2/10], Step [10000/12500], Loss: 2.0853
Epoch [2/10], Step [12000/12500], Loss: 2.1449
Epoch [3/10], Step [2000/12500], Loss: 1.9338
Epoch [3/10], Step [4000/12500], Loss: 1.6497
Epoch [3/10], Step [6000/12500], Loss: 1.7054
Epoch [3/10], Step [8000/12500], Loss: 1.6961
Epoch [3/10], Step [10000/12500], Loss: 1.0833
Epoch [3/10], Step [12000/12500], Loss: 2.1847
Epoch [4/10], Step [2000/12500], Loss: 3.0891
Epoch [4/10], Step [4000/12500], Loss: 1.7707
Epoch [4/10], Step [6000/12500], Loss: 1.6318
Epoch [4/10], Step [8000/125

In [18]:
with torch.no_grad():
    n_correct = 0
    n_samples = 0
    n_class_correct = [i for i in range(10)]
    n_samples_correct = [i for i in range(10)]
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        # max returns
        _, predicted = torch.max(outputs, 1)
        n_samples += labels.size(0)
        n_correct += (predicted == labels).sum().item()

        for i in range(batch_size):
            label = labels[i]
            pred = predicted[i]
            if (label == pred):
                n_class_correct[label] += 1
            n_samples_correct[label] += 1
    
    acc = 100.0 * (n_correct / n_samples)
    print(f'Accuracy of the network: {acc} %')

    for i in range(10):
        acc = 100.0 * (n_class_correct[i] / n_samples_correct[i])
        print(f'Accuracy of {classes[i]}: {acc} %')

Accuracy of the network: 57.97 %
Accuracy of plane: 65.4 %
Accuracy of car: 67.33266733266733 %
Accuracy of bird: 45.209580838323355 %
Accuracy of cat: 37.48753738783649 %
Accuracy of deer: 41.63346613545817 %
Accuracy of dog: 47.76119402985074 %
Accuracy of frog: 66.00397614314116 %
Accuracy of horse: 65.04468718967229 %
Accuracy of ship: 76.09126984126983 %
Accuracy of truck: 69.47472745292369 %
